# Prototipos funcionales — Detección y priorización de fraude

**Curso:** Inteligencia Artificial · Clases 14–15  
**Caso propio:** IBM TabFormer / proyecto iaupeu  
**Alcance:** exclusivamente la fase final del notebook del profesor: prototipo mínimo de inferencia, formulario ipywidgets y aplicación Gradio.

Se usan los artefactos de Unidad 2: el preprocesador entrenado y HistGradientBoostingClassifier. No se vuelve a entrenar, no se usa TensorFlow y no se necesitan los 24 millones de filas para realizar inferencia.

**Nota:** la puntuación de riesgo es un score sin calibración probabilística. Las recomendaciones son de apoyo humano, no acciones automáticas sobre tarjetas.

In [ ]:
# 0. CARGAR EL MODELO Y EL PREPROCESADOR EXISTENTES DE UNIDAD 2
from pathlib import Path
from datetime import date
import joblib
import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "artifacts/modelo_unidad2_seleccionado.joblib").is_file()
             and (p / "artifacts/preprocessor_unidad2.joblib").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Abre el notebook en una copia del repositorio iaupeu que contenga "
        "artifacts/modelo_unidad2_seleccionado.joblib y "
        "artifacts/preprocessor_unidad2.joblib."
    )

modelo = joblib.load(ROOT / "artifacts/modelo_unidad2_seleccionado.joblib")
preprocesador = joblib.load(ROOT / "artifacts/preprocessor_unidad2.joblib")
FEATURES_U2 = ["amount_log", "hour_sin", "hour_cos", "minute", "day_of_week",
               "is_weekend", "is_night", "Use Chip", "mcc_category", "is_online"]
MODALIDADES = ["Chip Transaction", "Swipe Transaction", "Online Transaction"]
print("Proyecto:", ROOT)
print("Clasificador:", type(modelo).__name__)
print("Preprocesador:", type(preprocesador).__name__)

## Fase 5 — Prototipo funcional de despliegue mínimo

Se ingresan datos normales de una transacción: importe, fecha, hora, modalidad de pago, si el comercio es online y código MCC. El prototipo reproduce las 10 características del clasificador original, incluido el mapeo de sectores comerciales.

Los importes no positivos siguen la regla OUT_OF_COHORT_REVIEW y no pasan al clasificador de compras.

In [ ]:
# 1. INGENIERÍA DE CARACTERÍSTICAS IGUAL A UNIDAD 2
def categoria_mcc(m):
    if 3000 <= m <= 3299: return "Airlines"
    if 3300 <= m <= 3499: return "Car_Rental"
    if 3500 <= m <= 3999: return "Lodging_Hotels"
    if 4000 <= m <= 4799: return "Transportation"
    if 4800 <= m <= 4999: return "Utilities_Telecom"
    if 5200 <= m <= 5499: return "Supermarkets_Groceries"
    if 5500 <= m <= 5599: return "Automotive_Fuel"
    if 5600 <= m <= 5699: return "Apparel_Clothing"
    if 5800 <= m <= 5899: return "Restaurants_Dining"
    if 5900 <= m <= 5999: return "Misc_Retail"
    if 6000 <= m <= 6999: return "Financial_Services"
    if 7000 <= m <= 7999: return "Entertainment_Recreation"
    if 8000 <= m <= 8999: return "Professional_Health"
    return "Other"

def construir_transaccion(monto, fecha_iso, hora, minuto, modalidad, canal, mcc):
    f = date.fromisoformat(str(fecha_iso))
    if modalidad not in MODALIDADES:
        raise ValueError("Modalidad de pago no permitida.")
    if not 0 <= int(hora) <= 23 or not 0 <= int(minuto) <= 59:
        raise ValueError("Hora y minuto fuera del rango válido.")
    if not 0 <= int(mcc) <= 9999 or float(mcc) != int(mcc):
        raise ValueError("MCC debe ser un entero entre 0 y 9999.")
    if not np.isfinite(float(monto)):
        raise ValueError("Importe no válido.")
    return {"Amount": float(monto), "Year": f.year, "Month": f.month, "Day": f.day,
            "Time": f"{int(hora):02d}:{int(minuto):02d}",
            "Use Chip": modalidad, "Merchant State": canal, "MCC": int(mcc)}

def extraer_features(datos):
    filas = []
    for _, r in datos.iterrows():
        f = date(int(r["Year"]), int(r["Month"]), int(r["Day"]))
        h, minute = map(int, str(r["Time"]).split(":"))
        if not 0 <= h <= 23 or not 0 <= minute <= 59:
            raise ValueError("Hora o minuto fuera de rango.")
        d = f.weekday() + 1       # Lunes=1 ... Domingo=7
        amt = float(r["Amount"])
        mcc = int(r["MCC"])
        online = str(r["Merchant State"]).strip().upper() == "ONLINE" or r["Use Chip"] == "Online Transaction"
        filas.append({
            "amount_log": np.log1p(max(amt, 0.0)),
            "hour_sin": np.sin(2 * np.pi * h / 24.0),
            "hour_cos": np.cos(2 * np.pi * h / 24.0),
            "minute": minute,
            "day_of_week": d,
            "is_weekend": int(d >= 6),
            "is_night": int(h <= 5),
            "Use Chip": r["Use Chip"],
            "mcc_category": categoria_mcc(mcc),
            "is_online": "ONLINE" if online else "IN_PERSON"
        })
    return pd.DataFrame(filas, columns=FEATURES_U2)

In [ ]:
# 2. INFERENCIA, PRIORIDAD Y RECOMENDACIONES
def pipeline_inferencia_y_recomendacion(transacciones):
    datos = transacciones.reset_index(drop=True).copy()
    resultados = []
    for i, r in datos.iterrows():
        amt = float(r["Amount"])
        if amt <= 0:
            resultados.append({
                "ID": i + 1, "Importe_USD": amt, "Risk_Score": None,
                "Prediccion_05": -1, "Prioridad": "OUT_OF_COHORT_REVIEW",
                "Recomendacion": "Revisión contable manual; fuera del modelo de compras."
            })
            continue
        X = extraer_features(pd.DataFrame([r]))
        X_procesado = preprocesador.transform(X)
        score = float(modelo.predict_proba(X_procesado)[0, 1])
        riesgo = score * 100
        if riesgo >= 70:
            nivel = "ALTA"
            accion = "Analista: revisar prioritariamente y verificar con titular."
        elif riesgo >= 40:
            nivel = "MEDIA"
            accion = "Analista: considerar autenticación adicional (2FA)."
        else:
            nivel = "BAJA"
            accion = "Seguimiento habitual bajo las reglas de control existentes."
        resultados.append({
            "ID": i + 1, "Importe_USD": amt,
            "Risk_Score": round(riesgo, 2),
            "Prediccion_05": int(score >= 0.50),
            "Prioridad": nivel, "Recomendacion": accion
        })
    return pd.DataFrame(resultados)

# Tres ejemplos didácticos; NO representan fraudes confirmados.
casos = pd.DataFrame([
    construir_transaccion(215.00, "2019-04-10", 13, 10, "Online Transaction", "ONLINE", 5999),
    construir_transaccion(44.00, "2019-06-17", 9, 35, "Chip Transaction", "CA", 5411),
    construir_transaccion(-45.00, "2019-11-02", 9, 15, "Chip Transaction", "CA", 5411),
])
display(pipeline_inferencia_y_recomendacion(casos))

## Opción A — Formulario dentro de Jupyter (ipywidgets)

El usuario llena datos de una sola transacción y pulsa Evaluar. El modelo transforma internamente los datos y devuelve un score y una recomendación. Requiere instalar ipywidgets.

In [ ]:
# 3. PROTOTIPO INTERACTIVO EN EL NOTEBOOK
import ipywidgets as widgets
from IPython.display import display, clear_output

monto_w = widgets.FloatText(value=146.52, description="Importe USD:", style={"description_width": "170px"})
fecha_w = widgets.DatePicker(value=date(2019, 4, 10), description="Fecha:", style={"description_width": "170px"})
hora_w = widgets.BoundedIntText(value=13, min=0, max=23, description="Hora:", style={"description_width": "170px"})
min_w = widgets.BoundedIntText(value=10, min=0, max=59, description="Minuto:", style={"description_width": "170px"})
chip_w = widgets.Dropdown(options=MODALIDADES, value="Online Transaction",
                          description="Uso de tarjeta:", style={"description_width": "170px"})
canal_w = widgets.Dropdown(options=[("Comercio online", "ONLINE"), ("Presencial", "CA")],
                           description="Comercio:", style={"description_width": "170px"})
mcc_w = widgets.BoundedIntText(value=5999, min=0, max=9999,
                               description="Código MCC:", style={"description_width": "170px"})
evaluar_w = widgets.Button(description="Evaluar transacción", button_style="primary", icon="search")
output_w = widgets.Output()

def evaluar_en_notebook(_):
    with output_w:
        clear_output(wait=True)
        try:
            if fecha_w.value is None:
                raise ValueError("Selecciona una fecha.")
            entrada = construir_transaccion(
                monto_w.value, fecha_w.value.isoformat(), hora_w.value,
                min_w.value, chip_w.value, canal_w.value, mcc_w.value
            )
            display(pipeline_inferencia_y_recomendacion(pd.DataFrame([entrada])))
            print("Score no calibrado. La decisión final corresponde al analista.")
        except Exception as exc:
            print("Error:", exc)

evaluar_w.on_click(evaluar_en_notebook)
display(widgets.VBox([
    widgets.HTML("<h3>Prototipo de fraude transaccional — IBM TabFormer</h3>"),
    monto_w, fecha_w, hora_w, min_w, chip_w, canal_w, mcc_w,
    evaluar_w, output_w
]))

## Opción B — Aplicación web con Gradio

Esta versión funciona con la misma lógica y los mismos dos archivos joblib. Si Gradio no está instalado, ejecuta antes: %pip install gradio

En local se inicia sin un enlace público, con share=False. No se necesita volver a entrenar.

In [ ]:
# 4. PROTOTIPO WEB: GRADIO
import gradio as gr

def predecir_interfaz(monto, fecha, hora, minuto, modalidad, canal, mcc):
    try:
        entrada = construir_transaccion(monto, fecha, hora, minuto, modalidad, canal, mcc)
        fila = pipeline_inferencia_y_recomendacion(pd.DataFrame([entrada])).iloc[0]
        score = "No aplica" if pd.isna(fila["Risk_Score"]) else f'{float(fila["Risk_Score"]):.2f} / 100'
        return score, str(fila["Prioridad"]), str(fila["Recomendacion"])
    except Exception as exc:
        return "Error", "Datos inválidos", str(exc)

interfaz_fraude = gr.Interface(
    fn=predecir_interfaz,
    inputs=[
        gr.Number(label="Importe de la transacción (USD)", value=146.52),
        gr.Textbox(label="Fecha YYYY-MM-DD", value="2019-04-10"),
        gr.Slider(0, 23, value=13, step=1, label="Hora"),
        gr.Slider(0, 59, value=10, step=1, label="Minuto"),
        gr.Dropdown(choices=MODALIDADES, value="Online Transaction", label="Modalidad de pago"),
        gr.Dropdown(choices=["ONLINE", "CA"], value="ONLINE", label="Canal (ONLINE / presencial)"),
        gr.Number(label="Código MCC", value=5999, precision=0)
    ],
    outputs=[
        gr.Textbox(label="Puntuación de riesgo, no calibrada"),
        gr.Textbox(label="Prioridad"),
        gr.Textbox(label="Recomendación para el analista")
    ],
    title="Sistema Inteligente de Detección y Priorización de Fraude",
    description="Prototipo académico IBM TabFormer con HistGradientBoosting. Solo apoyo a revisión humana.",
    examples=[
        [215.0, "2019-04-10", 13, 10, "Online Transaction", "ONLINE", 5999],
        [44.0, "2019-06-17", 9, 35, "Chip Transaction", "CA", 5411],
        [-45.0, "2019-11-02", 9, 15, "Chip Transaction", "CA", 5411]
    ]
)
interfaz_fraude.launch(share=False)

## Nota final

Solo se han adaptado las **secciones finales de prototipado** de la clase proporcionada. La inferencia es real cuando se ejecuta desde el repositorio con los artefactos originales. Los ejemplos no tienen etiquetas de fraude confirmadas y no representan métricas nuevas de evaluación.

Reglas conservadas del proyecto: Amount <= 0 va a OUT_OF_COHORT_REVIEW; score >= 70 = prioridad ALTA; 40–69.99 = MEDIA; por debajo de 40 = BAJA. La predicción binaria a umbral 0.5 es distinta de la clasificación de prioridad. Un score de 80 no equivale necesariamente a una probabilidad real de fraude de 80%.